In [1]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

prc = 100
a = -1
b = 1
x0 = 0
sig = 0.5
x1 = 1
sig1 = 1

A1 = np.random.uniform(low=a, high=b, size=prc)
B1 = np.random.uniform(low=a, high=b, size=prc)
E1 = np.random.uniform(low=a, high=b, size=prc)

tb1 = np.column_stack((A1 - 0.5, B1, 0.5 + A1 + B1 + E1)) #матрица 100 на 3

A2 = np.random.normal(x0, sig, prc)
B2 = np.random.normal(x0, sig, prc)
E2 = np.random.normal(x0, sig, prc)

tb2 = np.column_stack((A2, 0.1*B2, 0.1*A2 + 0.2* B2 + 0.1*E2)) #матрица 100 на 3

X = np.random.normal(x1, sig1, prc)
Y = np.random.uniform(a, b, prc) 

tb3 = np.column_stack((X, Y, X**2 + Y)) #матрица 100 на 3



fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=A1 - 0.5,
    y=B1,
    z=0.5 + A1 + B1 + E1,
    mode='markers',
    marker=dict(size=4, color='royalblue', opacity=0.8),
    name='prec 1'
))
fig.add_trace(go.Scatter3d(
    x=A2,
    y=0.1 * B2,
    z=0.1 * A2 + 0.2 * B2 + 0.1 * E2,
    mode='markers',
    marker=dict(size=4, color='darkorange', opacity=0.8),
    name='prec 2'
))
fig.add_trace(go.Scatter3d(
    x=X,
    y=Y,
    z=X**2 + Y,
    mode='markers',
    marker=dict(size=4, color='forestgreen', opacity=0.8),
    name='prec 3'
))
fig.update_layout(
    title='3D график трех выборок',
    scene=dict(
        xaxis_title='x',
        yaxis_title='y',
        zaxis_title='z'
    ),
    width=900,
    height=700,
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

lab1 = np.zeros(len(tb1))
lab2 = np.ones(len(tb2))
lab3 = np.full(len(tb3), 2)
Xt = np.vstack([tb1, tb2, tb3])
Yt = np.concatenate([lab1, lab2, lab3])

x_tr, xt, y_tr, yt = train_test_split(Xt, Yt, test_size=0.2, random_state=9) #парметр random_state определяет способ выбора 20 и 80 процентов из выборки. Генератор случайных числев будет стартовать с всегда с 9

scaler = StandardScaler()
x_tr_sc = scaler.fit_transform(x_tr)
xt_sc = scaler.fit_transform(xt)


In [3]:
from sklearn.svm import SVC
from sklearn.mixture import GaussianMixture
from sklearn.gaussian_process.kernels import RBF
from sklearn.gaussian_process import GaussianProcessClassifier

md1 = SVC(kernel='linear', C=0.7)
md1.fit(x_tr_sc, y_tr)

gss_0 = GaussianMixture(n_components=1, covariance_type='full', random_state=9).fit(x_tr_sc[y_tr == 0])
gss_1 = GaussianMixture(n_components=1, covariance_type='full', random_state=9).fit(x_tr_sc[y_tr == 1])
gss_2 = GaussianMixture(n_components=1, covariance_type='full', random_state=9).fit(x_tr_sc[y_tr == 2])

gss_0.fit(x_tr_sc[y_tr == 0])
gss_1.fit(x_tr_sc[y_tr == 1])
gss_2.fit(x_tr_sc[y_tr == 2])

ker = 1.0 * RBF(length_scale=1.0)

gpc = GaussianProcessClassifier(kernel=ker, random_state=9)
gpc.fit(x_tr_sc, y_tr)


,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ""1.0 * RBF(1.0)"" is used as default. Note thatthe kernel's hyperparameters are optimized during fitting. Also kernelcannot be a `CompoundKernel`.",1**2 * RBF(length_scale=1)
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:`Glossary <random_state>`.",9
,kernel__k1,1**2
,kernel__k2,RBF(length_scale=1)
,kernel__k1__constant_value,1.0
,kernel__k1__constant_value_bounds,"(1e-05, ...)"
,kernel__k2__length_scale,1.0
,kernel__k2__length_scale_bounds,"(1e-05, ...)"
,"optimizer optimizer: 'fmin_l_bfgs_b', callable or None, default='fmin_l_bfgs_b'Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func' is the objective function to be maximized, which # takes the hyperparameters theta as parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the 'L-BFGS-B' algorithm from scipy.optimize.minimizeis used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are:: 'fmin_l_bfgs_b'",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that n_restarts_optimizer=0 implies that onerun is performed.",0
,"max_iter_predict max_iter_predict: int, default=100The maximum number of iterations in Newton's method for approximatingthe posterior during predict. Smaller values will reduce computationtime at the cost of worse results.",100


In [4]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

y_pr = md1.predict(xt_sc)
prec = precision_score(yt, y_pr, average='macro')
rec = recall_score(yt, y_pr, average='macro')

w = md1.coef_[0]
b = md1.intercept_[0]

p0 = np.mean(y_tr == 0)
p1 = np.mean(y_tr == 1)
p2 = np.mean(y_tr == 2)

log_p0 = gss_0.score_samples(xt_sc) + np.log(p0)
log_p1 = gss_1.score_samples(xt_sc) + np.log(p1)
log_p2 = gss_2.score_samples(xt_sc) + np.log(p2)

# Выбираем тот класс (0, 1 или 2), у которого логарифм вероятности самый большой:
yg_pr = np.argmax(np.column_stack([log_p0, log_p1, log_p2]), axis=1)

prec_g = precision_score(yt, yg_pr, average='macro')
rec_g = recall_score(yt, yg_pr, average='macro')

ygc_pr = gpc.predict(xt_sc)

prec_gp = precision_score(yt, ygc_pr, average='macro')
rec_gp = recall_score(yt, ygc_pr, average='macro')
probs = gpc.predict_proba(xt_sc)[:, 1]



In [7]:
from itertools import combinations
x_min, x_max = x_tr_sc[:, 0].min(), x_tr_sc[:, 0].max()
y_min, y_max = x_tr_sc[:, 1].min(), x_tr_sc[:, 1].max()
z_min, z_max = x_tr_sc[:, 2].min(), x_tr_sc[:, 2].max()

#xx, yy = np.meshgrid(np.linspace(x_min, x_max, 25), np.linspace(y_min, y_max, 25))
#zz = -(w[0] * xx + w[1] * yy + b) / w[2]


fig = go.Figure()


fig.add_trace(go.Scatter3d(
    x=x_tr_sc[y_tr == 0, 0],
    y=x_tr_sc[y_tr == 0, 1],
    z=x_tr_sc[y_tr == 0, 2],
    mode='markers',
    marker=dict(size=4, color='royalblue', opacity=0.7),
    name='Класс 0 (Выборка 1)'
))

fig.add_trace(go.Scatter3d(
    x=x_tr_sc[y_tr == 1, 0],
    y=x_tr_sc[y_tr == 1, 1],
    z=x_tr_sc[y_tr == 1, 2],
    mode='markers',
    marker=dict(size=4, color='darkorange', opacity=0.7),
    name='Класс 1 (Выборки 2)'
))

fig.add_trace(go.Scatter3d(
    x=x_tr_sc[y_tr == 2, 0],
    y=x_tr_sc[y_tr == 2, 1],
    z=x_tr_sc[y_tr == 2, 2],
    mode='markers',
    marker=dict(size=4, color='green', opacity=0.7),
    name='Класс 2 (Выборки 3)'
))

classes = md1.classes_
pairs = list(combinations(range(len(classes)), 2))

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 30),
    np.linspace(y_min, y_max, 30)
)

for i in range(len(md1.coef_)):
    w = md1.coef_[i]
    b = md1.intercept_[i]

    c1_idx, c2_idx = pairs[i]
    c1 = classes[c1_idx]
    c2 = classes[c2_idx]
    colors = ['red', 'green', 'blue']

    print(f"\nПлоскость {i}: класс {c1} vs класс {c2}")
    print(f"  w = {w}")
    print(f"  b = {b}")
    print(f"  Уравнение: {w[0]:.3f}*x1 + {w[1]:.3f}*x2 + {w[2]:.3f}*x3 + ({b:.3f}) = 0")

    # --- Вычисляем zz = z(x,y) ---
    if abs(w[2]) > 1e-6:
        zz = -(w[0] * xx + w[1] * yy + b) / w[2]
        zz = np.clip(zz, z_min, z_max)

        fig.add_trace(go.Surface(
            x=xx,
            y=yy,
            z=zz,
            opacity=0.4,
            colorscale=[[0, colors[i]], [1, colors[i]]],
            showscale=False,
            name=f'{c1} vs {c2}',
            showlegend=True
        ))
    else:
        # Плоскость параллельна оси Z — выражаем x через y,z
        print(f"  ⚠ w[2]≈0 → плоскость вертикальная, строим через x(y,z)")
        yy2, zz2 = np.meshgrid(
            np.linspace(y_min, y_max, 30),
            np.linspace(z_min, z_max, 30)
        )
        if abs(w[0]) > 1e-6:
            xx2 = -(w[1] * yy2 + b) / w[0]
        else:
            xx2 = -(w[1] * yy2 + w[2] * zz2 + b) / 1e-6  # fallback
        xx2 = np.clip(xx2, x_min, x_max)

        fig.add_trace(go.Surface(
            x=xx2,
            y=yy2,
            z=zz2,
            opacity=0.4,
            colorscale=[[0, colors[i]], [1, colors[i]]],
            showscale=False,
            name=f'{c1} vs {c2}',
            showlegend=True
        ))

fig.add_trace(go.Surface(
    x=xx,
    y=yy,
    z=zz,
    opacity=0.4,
    colorscale=[[0, 'gray'], [1, 'gray']],  # Серый цвет плоскости
    showscale=False,                        # Скрыть цветовую шкалу
    name='Плоскость SVM',
    showlegend=True
))

fig.update_layout(
    title=f'SVM на отмасштабированных данных<br><sup>precision = {prec:.4f}, recall = {rec:.4f}</sup>',
    scene=dict(
        xaxis_title='X (scaled)',
        yaxis_title='Y (scaled)',
        zaxis_title='Z (scaled)'
    ),
    width=900,
    height=700,
    margin=dict(l=0, r=0, b=0, t=50)
)


fig.show()

# Вывод метрик в консоль
print(f'precision = {prec:.6f}, recall = {rec:.6f}')


Плоскость 0: класс 0.0 vs класс 1.0
  w = [-1.7828271  -0.85683866  2.69505488]
  b = 0.13519281486399293
  Уравнение: -1.783*x1 + -0.857*x2 + 2.695*x3 + (0.135) = 0

Плоскость 1: класс 0.0 vs класс 2.0
  w = [-1.95754694 -0.29730425  1.02432623]
  b = 0.051977921570170754
  Уравнение: -1.958*x1 + -0.297*x2 + 1.024*x3 + (0.052) = 0

Плоскость 2: класс 1.0 vs класс 2.0
  w = [ 0.00745284  1.06660707 -3.44524418]
  b = -0.6548564405458748
  Уравнение: 0.007*x1 + 1.067*x2 + -3.445*x3 + (-0.655) = 0


precision = 0.830808, recall = 0.793016


In [11]:
def get_dome_mesh(mean, cov, n_std=1.8, n_pts=30):
    vals, vecs = np.linalg.eigh(cov)
    u = np.linspace(0, 2 * np.pi, n_pts)
    v = np.linspace(0, np.pi, n_pts)
    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones_like(u), np.cos(v))
    sphere = np.stack([x, y, z], axis=-1)
    radii = n_std * np.sqrt(np.maximum(vals, 0))
    dome = (sphere * radii) @ vecs.T + mean
    return dome[:, :, 0], dome[:, :, 1], dome[:, :, 2]

# Расчет куполов для трех классов
x0, y0, z0 = get_dome_mesh(gss_0.means_[0], gss_0.covariances_[0])
x1, y1, z1 = get_dome_mesh(gss_1.means_[0], gss_1.covariances_[0])
x2, y2, z2 = get_dome_mesh(gss_2.means_[0], gss_2.covariances_[0])

fig3 = go.Figure()

# Точки 3 классов
fig3.add_trace(go.Scatter3d(x=x_tr_sc[y_tr == 0, 0], y=x_tr_sc[y_tr == 0, 1], z=x_tr_sc[y_tr == 0, 2], mode='markers', marker=dict(size=4, color='royalblue', opacity=0.7), name='Класс 0 (Выборка 1)'))
fig3.add_trace(go.Scatter3d(x=x_tr_sc[y_tr == 1, 0], y=x_tr_sc[y_tr == 1, 1], z=x_tr_sc[y_tr == 1, 2], mode='markers', marker=dict(size=4, color='darkorange', opacity=0.7), name='Класс 1 (Выборка 2)'))
fig3.add_trace(go.Scatter3d(x=x_tr_sc[y_tr == 2, 0], y=x_tr_sc[y_tr == 2, 1], z=x_tr_sc[y_tr == 2, 2], mode='markers', marker=dict(size=4, color='forestgreen', opacity=0.7), name='Класс 2 (Выборка 3)'))

# 3 купола
fig3.add_trace(go.Surface(x=x0, y=y0, z=z0, colorscale=[[0, 'blue'], [1, 'blue']], opacity=0.25, showscale=False, name='Купол 0'))
fig3.add_trace(go.Surface(x=x1, y=y1, z=z1, colorscale=[[0, 'orange'], [1, 'orange']], opacity=0.25, showscale=False, name='Купол 1'))
fig3.add_trace(go.Surface(x=x2, y=y2, z=z2, colorscale=[[0, 'green'], [1, 'green']], opacity=0.25, showscale=False, name='Купол 2'))

fig3.update_layout(title=f'GMM: 3 класса<br><sup>precision = {prec_g:.4f}, recall = {rec_g:.4f}</sup>', scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'), width=900, height=650)
fig3.show()

In [12]:
def create_rbf_dome(data, n_std=1.8, n_pts=35):
    mean = np.mean(data, axis=0)
    cov = np.cov(data, rowvar=False)
    vals, vecs = np.linalg.eigh(cov)
    
    u = np.linspace(0, 2 * np.pi, n_pts)
    v = np.linspace(0, np.pi, n_pts)
    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones_like(u), np.cos(v))
    sphere = np.stack([x, y, z], axis=-1)
    
    radii = n_std * np.sqrt(np.maximum(vals, 0))
    dome = (sphere * radii) @ vecs.T + mean
    return dome[:, :, 0], dome[:, :, 1], dome[:, :, 2]


# 1. Выделяем точки каждого из 3 классов для построения куполов
pts_c0 = x_tr_sc[y_tr == 0]  # Класс 0 (Выборка 1)
pts_c1 = x_tr_sc[y_tr == 1]  # Класс 1 (Выборка 2)
pts_c2 = x_tr_sc[y_tr == 2]  # Класс 2 (Выборка 3)

# 2. Считаем форму 3 куполов (функция create_rbf_dome у вас уже объявлена)
x_d0, y_d0, z_d0 = create_rbf_dome(pts_c0, n_std=1.8)
x_d1, y_d1, z_d1 = create_rbf_dome(pts_c1, n_std=1.8)
x_d2, y_d2, z_d2 = create_rbf_dome(pts_c2, n_std=1.8)

# ==============================================================
# 3. 3D-график в Plotly для Гауссова процесса (GPC)
# ==============================================================
fig = go.Figure()

# --- Точки трех классов (по предсказанию модели GPC: ygc_pr) ---
fig.add_trace(go.Scatter3d(
    x=xt_sc[ygc_pr == 0, 0], y=xt_sc[ygc_pr == 0, 1], z=xt_sc[ygc_pr == 0, 2],
    mode='markers', marker=dict(size=4, color='royalblue', opacity=0.8),
    name='Класс 0 (предсказано: Выборка 1)'
))

fig.add_trace(go.Scatter3d(
    x=xt_sc[ygc_pr == 1, 0], y=xt_sc[ygc_pr == 1, 1], z=xt_sc[ygc_pr == 1, 2],
    mode='markers', marker=dict(size=4, color='darkorange', opacity=0.8),
    name='Класс 1 (предсказано: Выборка 2)'
))

fig.add_trace(go.Scatter3d(
    x=xt_sc[ygc_pr == 2, 0], y=xt_sc[ygc_pr == 2, 1], z=xt_sc[ygc_pr == 2, 2],
    mode='markers', marker=dict(size=4, color='forestgreen', opacity=0.8),
    name='Класс 2 (предсказано: Выборка 3)'
))

# --- Объемные RBF-купола для каждого из 3 классов ---
fig.add_trace(go.Surface(
    x=x_d0, y=y_d0, z=z_d0,
    colorscale=[[0, 'blue'], [1, 'blue']], opacity=0.25,
    showscale=False, name='RBF-купол (Класс 0)', showlegend=True
))

fig.add_trace(go.Surface(
    x=x_d1, y=y_d1, z=z_d1,
    colorscale=[[0, 'orange'], [1, 'orange']], opacity=0.25,
    showscale=False, name='RBF-купол (Класс 1)', showlegend=True
))

fig.add_trace(go.Surface(
    x=x_d2, y=y_d2, z=z_d2,
    colorscale=[[0, 'green'], [1, 'green']], opacity=0.25,
    showscale=False, name='RBF-купол (Класс 2)', showlegend=True
))

# --- Заголовок с метриками ---
fig.update_layout(
    title=f'Гауссов процесс с RBF-ядром (3 класса)<br><sup>precision = {prec_gp:.4f}, recall = {rec_gp:.4f}</sup>',
    scene=dict(
        xaxis_title='X (scaled)',
        yaxis_title='Y (scaled)',
        zaxis_title='Z (scaled)'
    ),
    width=950,
    height=750,
    margin=dict(l=0, r=0, b=0, t=60)
)

fig.show()